In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

HERE = Path.cwd()
OUT_CSV = HERE / "home_advantage_by_league.csv"

actual_files = sorted(HERE.glob("*_actual.csv"))

In [5]:
def infer_results(df: pd.DataFrame) -> pd.Series:
    cols = {c.lower(): c for c in df.columns}

    if "hometeamresult" in cols:
        res_col = cols["hometeamresult"]
        s = pd.to_numeric(df[res_col], errors="coerce")

        mapped = s.map({1: "H", 0: "D", -1: "A"})

        if mapped.notna().all():
            return mapped.astype("string")
        
    if "hometeamgoals" in cols and "awayteamgoals" in cols:
        hg = pd.to_numeric(df[cols["hometeamgoals"]], errors="coerce")
        ag = pd.to_numeric(df[cols["awayteamgoals"]], errors="coerce")
        out = pd.Series(index=df.index, dtype="string")
        out[hg > ag]  = "H"
        out[hg < ag]  = "A"
        out[hg == ag] = "D"
        return out

In [8]:
def compute_home_advantage_for_file(csv_path: Path) -> dict:
    df = pd.read_csv(csv_path)
    results = infer_results(df)
    total   = results.size

    h = int((results == "H").sum())
    d = int((results == "D").sum())
    a = int((results == "A").sum())

    home_equiv = h + d/3.0

    p_home = (home_equiv / total) if total > 0 else 0.0

    league = csv_path.name.replace("_actual.csv", "")

    return {
        "league": league,
        "matches_total": total,
        "home_wins": h,
        "draws": d,
        "away_wins": a,
        "home_win_with_draws": round(home_equiv, 6),
        "home_win_pct": round(100.0 * p_home, 2)
    }

In [9]:
rows = [compute_home_advantage_for_file(p) for p in actual_files]
home_adv_df = pd.DataFrame(rows).sort_values("league").reset_index(drop=True)
home_adv_df

,league,matches_total,home_wins,draws,away_wins,home_win_with_draws,home_win_pct
0,bundesliga,6426,2883,1601,1942,3416.666667,53.17
1,la_liga,8740,4105,2199,2436,4838.000000,55.35
2,premier_league,8360,3826,2028,2506,4502.000000,53.85
3,serie_a,8518,3811,2278,2429,4570.333333,53.66


In [10]:
home_adv_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

Saved: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/data/european_soccer_leagues/actual/home_advantage_by_league.csv
